# Exercises XP – Diabetes Classification

## Ce qu'on va apprendre
- Comprendre un problème de classification binaire
- Charger, explorer et visualiser un dataset médical
- Entraîner et comparer plusieurs modèles de classification
- Évaluer un modèle (accuracy, matrice de confusion, précision, rappel, F1, ROC)
- Gérer le déséquilibre de classes
- Optimiser le seuil de décision

**Dataset :** Diabetes Prediction Dataset (100 000 patients)

## Exercise 1 – Comprendre le problème et charger les données

On veut **prédire si un individu est diabétique** (1) ou non (0) à partir de ses caractéristiques médicales.  
C'est un problème de **classification binaire supervisée**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

%matplotlib inline
sns.set_theme(style='whitegrid')

df = pd.read_csv('diabetes_prediction_dataset.csv')

print('Dimensions :', df.shape)
display(df.head())
print('\nTypes des colonnes :')
print(df.dtypes)
print('\nValeurs manquantes par colonne :')
display(df.isna().sum().sort_values(ascending=False))

In [ ]:
assert 'diabetes' in df.columns, "Expected a 'diabetes' target column"

counts = df['diabetes'].value_counts()
print('Répartition des cas :')
print(f'  Négatifs (0 – sans diabète) : {counts[0]:,}')
print(f'  Positifs (1 – avec diabète) : {counts[1]:,}')
print(f'  Taux de diabète             : {counts[1] / len(df) * 100:.1f}%')

plt.figure(figsize=(6, 4))
sns.countplot(x='diabetes', data=df, palette=['#2ecc71', '#e74c3c'])
plt.title('Répartition des cas (0 = non diabétique, 1 = diabétique)')
plt.xlabel('Diabète')
plt.ylabel('Nombre de patients')
plt.xticks([0, 1], ['Non diabétique', 'Diabétique'])
plt.tight_layout()
plt.show()

In [ ]:
# Distribution des features numériques par classe (diabétique vs non)
num_features = ['age', 'bmi', 'HbA1c_level', 'blood_glucose_level']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i, col in enumerate(num_features):
    axes[0, i].hist(df[col], bins=40, color='steelblue', alpha=0.7, edgecolor='white')
    axes[0, i].set_title(f'Distribution – {col}', fontsize=9)

    df[df['diabetes'] == 0][col].hist(bins=40, ax=axes[1, i], alpha=0.6,
                                       color='#2ecc71', label='0', edgecolor='white')
    df[df['diabetes'] == 1][col].hist(bins=40, ax=axes[1, i], alpha=0.6,
                                       color='#e74c3c', label='1', edgecolor='white')
    axes[1, i].set_title(f'{col} par classe', fontsize=9)
    axes[1, i].legend(title='Diabète')

plt.suptitle('Distributions par feature (vert = non diabétique, rouge = diabétique)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Matrice de corrélation
corr_cols = num_features + ['hypertension', 'heart_disease', 'diabetes']
corr = df[corr_cols].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5, vmin=-1, vmax=1)
plt.title('Matrice de corrélation', fontweight='bold')
plt.tight_layout()
plt.show()

print('Corrélation avec la cible (diabetes) :')
print(corr['diabetes'].sort_values(ascending=False))

In [ ]:
X = df.drop(columns=['diabetes'])
y = df['diabetes']

# Stratify=y pour conserver la proportion de diabétiques dans les deux sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Taille du train :', X_train.shape)
print('Taille du test  :', X_test.shape)
print('\nProportion de diabétiques dans le train :', y_train.mean().round(4))
print('Proportion de diabétiques dans le test  :', y_test.mean().round(4))

## Exercise 2 – Choix du modèle et standardisation

### Quel modèle choisir ?

On utilise la **Régression Logistique** pour plusieurs raisons :
- Conçue pour la **classification binaire** (sortie entre 0 et 1)
- Produit des **probabilités calibrées** : on peut dire "ce patient a 73% de risque de diabète"
- **Interprétable** : chaque coefficient indique l'importance d'une variable
- Établit une **frontière de décision linéaire**, adaptée quand les classes sont séparables

### Faut-il standardiser ?

**Oui.** La régression logistique utilise un optimiseur sensible à l'échelle des variables. Sans standardisation :
- L'âge (18–90) et le BMI (15–50) n'ont pas la même échelle que le glucose (70–300)
- Le modèle converge moins bien et les coefficients sont difficilement comparables

On applique `StandardScaler` sur les colonnes numériques et `OneHotEncoder` sur les colonnes catégorielles.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print('Colonnes catégorielles :', cat_cols)
print('Colonnes numériques    :', num_cols)

preprocess = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
    ('num', StandardScaler(), num_cols)
])

print('\nPréprocesseur créé avec succès.')

## Exercise 3 – Entraînement du modèle

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

clf = Pipeline([
    ('pre', preprocess),
    ('lr',  LogisticRegression(max_iter=1000, random_state=42))
])

clf.fit(X_train, y_train)
print('Modèle entraîné avec succès.')
print(f'Nombre d\'itérations convergées : {clf.named_steps["lr"].n_iter_[0]}')

## Exercise 4 – Métriques d'évaluation

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, ConfusionMatrixDisplay
)

y_pred = clf.predict(X_test)

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)

print('=== Métriques sur le jeu de test ===')
print(f'Accuracy  (exactitude) : {acc:.4f}')
print(f'Precision              : {prec:.4f}')
print(f'Recall    (rappel)     : {rec:.4f}')
print(f'F1-Score               : {f1:.4f}')

metrics_names  = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
metrics_values = [acc, prec, rec, f1]
bar_colors     = ['#3498db', '#2ecc71', '#e67e22', '#9b59b6']

plt.figure(figsize=(8, 5))
bars = plt.bar(metrics_names, metrics_values, color=bar_colors, edgecolor='white')
plt.ylim(0, 1.1)
plt.title('Métriques – Logistic Regression (test set)', fontsize=12, fontweight='bold')
plt.ylabel('Score')
for bar, val in zip(bars, metrics_values):
    plt.text(bar.get_x() + bar.get_width() / 2, val + 0.02,
             f'{val:.3f}', ha='center', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Non diabétique', 'Diabétique'])

fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Matrice de Confusion', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'Vrais Négatifs  (TN) : {tn:,}  – patients sains correctement identifiés')
print(f'Faux Positifs   (FP) : {fp:,}  – patients sains prédits diabétiques (fausse alarme)')
print(f'Faux Négatifs   (FN) : {fn:,}  – patients diabétiques manqués (dangereux !)')
print(f'Vrais Positifs  (TP) : {tp:,}  – patients diabétiques correctement identifiés')

### Commentaire des métriques

- **Accuracy (~96%)** : Élevée, mais trompeuse sur un dataset déséquilibré (91.5% non-diabétiques). Un modèle naïf qui prédit toujours 0 obtiendrait déjà 91.5%.
- **Precision** : Sur les patients prédits diabétiques, combien le sont vraiment ? Haute précision = peu de fausses alarmes.
- **Recall** : Sur les vrais diabétiques, combien sont détectés ? En médecine, c'est la métrique **la plus critique** — un faux négatif (diabétique non détecté) est dangereux.
- **F1-Score** : Équilibre entre Precision et Recall — métrique la plus pertinente pour les classes déséquilibrées.
- **Faux Négatifs** : Le nombre de diabétiques manqués est le point critique à surveiller.

## Exercise 5 – Visualisation de la frontière de décision

In [ ]:
feat_x = 'HbA1c_level'
feat_y = 'blood_glucose_level'

X2_train = X_train[[feat_x, feat_y]].copy()
X2_test  = X_test[[feat_x, feat_y]].copy()

# Modèle entraîné sur 2 features pour la visualisation 2D
pipe2 = Pipeline([
    ('scaler', StandardScaler()),
    ('lr',     LogisticRegression(max_iter=1000, random_state=42))
])
pipe2.fit(X2_train.values, y_train)

x_min, x_max = X2_train[feat_x].min() - 0.5, X2_train[feat_x].max() + 0.5
y_min, y_max = X2_train[feat_y].min() - 5,   X2_train[feat_y].max() + 5
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 300),
    np.linspace(y_min, y_max, 300)
)

probs = pipe2.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1].reshape(xx.shape)
acc2  = accuracy_score(y_test, pipe2.predict(X2_test.values))

plt.figure(figsize=(10, 7))
plt.contourf(xx, yy, probs, levels=20, cmap='RdYlGn_r', alpha=0.6)
plt.colorbar(label='Probabilité de diabète')
plt.contour(xx, yy, probs, levels=[0.5], colors='black', linewidths=2)

sample_idx = np.random.choice(len(X2_test), size=800, replace=False)
plt.scatter(
    X2_test[feat_x].values[sample_idx],
    X2_test[feat_y].values[sample_idx],
    c=y_test.values[sample_idx],
    cmap='bwr', edgecolors='k', linewidth=0.3, s=25, alpha=0.8
)

plt.xlabel(f'{feat_x}  (taux HbA1c)', fontsize=11)
plt.ylabel(f'{feat_y}  (glycémie)', fontsize=11)
plt.title(f'Frontière de décision – 2 features | Accuracy test : {acc2:.3f}',
          fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Bleu = non diabétique   |   Rouge = diabétique')
print('Ligne noire = frontière de décision (P = 0.5)')

## Exercise 6 – Courbe ROC

In [ ]:
from sklearn import metrics

y_proba = clf.predict_proba(X_test)[:, 1]
fpr, tpr, _ = metrics.roc_curve(y_test, y_proba)
auc = metrics.roc_auc_score(y_test, y_proba)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='steelblue', linewidth=2.5,
         label=f'Logistic Regression (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', linewidth=1,
         label='Modèle aléatoire (AUC = 0.5)')
plt.fill_between(fpr, tpr, alpha=0.1, color='steelblue')
plt.xlabel('Taux de Faux Positifs (FPR)', fontsize=11)
plt.ylabel('Taux de Vrais Positifs (TPR)', fontsize=11)
plt.title('Courbe ROC – Régression Logistique', fontsize=13, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'AUC : {auc:.4f}')

### Interprétation de la courbe ROC

- La **courbe ROC** trace le compromis TPR/FPR pour tous les seuils de décision possibles.
- **AUC** mesure la capacité à distinguer les deux classes : 1.0 = parfait, 0.5 = aléatoire.
- Une AUC > 0.97 confirme que le modèle distingue très bien diabétiques et non-diabétiques.
- En abaissant le seuil (ex. 0.3), on augmente le rappel au prix d'un peu plus de fausses alarmes — ce qui est souvent préférable en médecine.

---
## Aller plus loin – Comparaison de modèles, déséquilibre et optimisation

Les exercises précédents ont utilisé uniquement la régression logistique. On va maintenant :
- Comparer 3 modèles de classification
- Gérer le déséquilibre de classes
- Optimiser le seuil de décision pour un usage médical
- Analyser l'importance des features

### Comparaison de 3 modèles : Logistic Regression, KNN, Random Forest

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'KNN (k=5)':           KNeighborsClassifier(n_neighbors=5),
    'Random Forest':        RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
}

results       = []
trained_pipes = {}

for model_name, model in models.items():
    pipe = Pipeline([('pre', preprocess), ('clf', model)])
    pipe.fit(X_train, y_train)
    trained_pipes[model_name] = pipe

    y_pred_m  = pipe.predict(X_test)
    y_proba_m = pipe.predict_proba(X_test)[:, 1]

    results.append({
        'Modèle':    model_name,
        'Accuracy':  round(accuracy_score(y_test, y_pred_m),  4),
        'Precision': round(precision_score(y_test, y_pred_m), 4),
        'Recall':    round(recall_score(y_test, y_pred_m),    4),
        'F1-Score':  round(f1_score(y_test, y_pred_m),        4),
        'AUC-ROC':   round(metrics.roc_auc_score(y_test, y_proba_m), 4)
    })

results_df = pd.DataFrame(results)
print('=== Comparaison des modèles ===')
display(results_df)

In [ ]:
# Graphique comparatif
metrics_cols  = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']
x             = np.arange(len(metrics_cols))
width         = 0.25
model_colors  = ['#3498db', '#e67e22', '#2ecc71']

plt.figure(figsize=(13, 6))
for i, row in results_df.iterrows():
    values = [row[m] for m in metrics_cols]
    plt.bar(x + i * width, values, width, label=row['Modèle'],
            color=model_colors[i], alpha=0.85, edgecolor='white')

plt.xticks(x + width, metrics_cols)
plt.ylim(0, 1.12)
plt.ylabel('Score')
plt.title('Comparaison des métriques – 3 modèles', fontsize=13, fontweight='bold')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### Gestion du déséquilibre de classes

Le dataset est déséquilibré : **91.5% non diabétiques vs 8.5% diabétiques**.  
L'option `class_weight='balanced'` force le modèle à accorder plus d'importance aux cas rares (diabétiques).

In [ ]:
from sklearn.metrics import classification_report

clf_std = Pipeline([
    ('pre', preprocess),
    ('lr',  LogisticRegression(max_iter=1000, random_state=42))
])
clf_std.fit(X_train, y_train)

clf_bal = Pipeline([
    ('pre', preprocess),
    ('lr',  LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'))
])
clf_bal.fit(X_train, y_train)

print('=== Logistic Regression STANDARD ===')
print(classification_report(y_test, clf_std.predict(X_test),
                             target_names=['Non diabétique', 'Diabétique']))

print('=== Logistic Regression BALANCED (class_weight) ===')
print(classification_report(y_test, clf_bal.predict(X_test),
                             target_names=['Non diabétique', 'Diabétique']))

### Optimisation du seuil de décision

Par défaut, le seuil est à **0.5** (si P ≥ 0.5 → diabétique).  
En médecine, on préfère souvent abaisser ce seuil pour détecter plus de cas positifs, même au prix de quelques fausses alarmes supplémentaires.

In [ ]:
# Courbes ROC superposées des 3 modèles
plt.figure(figsize=(9, 7))
line_colors = ['#3498db', '#e67e22', '#2ecc71']

for (model_name, pipe), color in zip(trained_pipes.items(), line_colors):
    y_proba_m = pipe.predict_proba(X_test)[:, 1]
    fpr_m, tpr_m, _ = metrics.roc_curve(y_test, y_proba_m)
    auc_m = metrics.roc_auc_score(y_test, y_proba_m)
    plt.plot(fpr_m, tpr_m, linewidth=2, color=color,
             label=f'{model_name} (AUC = {auc_m:.3f})')

plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Modèle aléatoire')
plt.xlabel('Taux de Faux Positifs (FPR)', fontsize=11)
plt.ylabel('Taux de Vrais Positifs (TPR)', fontsize=11)
plt.title('Courbes ROC – Comparaison des 3 modèles', fontsize=13, fontweight='bold')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Optimisation du seuil sur le Random Forest (meilleur AUC)
y_proba_rf = trained_pipes['Random Forest'].predict_proba(X_test)[:, 1]

thresholds  = np.arange(0.1, 0.9, 0.05)
recalls, precisions, f1s = [], [], []

for thresh in thresholds:
    y_pred_t = (y_proba_rf >= thresh).astype(int)
    recalls.append(recall_score(y_test, y_pred_t))
    precisions.append(precision_score(y_test, y_pred_t, zero_division=0))
    f1s.append(f1_score(y_test, y_pred_t))

best_thresh = thresholds[np.argmax(f1s)]

plt.figure(figsize=(10, 5))
plt.plot(thresholds, recalls,    marker='o', label='Recall',    color='#e74c3c')
plt.plot(thresholds, precisions, marker='s', label='Precision', color='#3498db')
plt.plot(thresholds, f1s,        marker='^', label='F1-Score',  color='#2ecc71', linewidth=2.5)
plt.axvline(x=best_thresh, color='black', linestyle='--', alpha=0.6,
            label=f'Seuil optimal F1 = {best_thresh:.2f}')
plt.xlabel('Seuil de décision')
plt.ylabel('Score')
plt.title('Impact du seuil sur les métriques (Random Forest)', fontsize=12, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Seuil optimal (max F1) : {best_thresh:.2f}')
print(f'F1 au seuil optimal    : {f1s[np.argmax(f1s)]:.4f}')
print(f'Recall au seuil optimal: {recalls[np.argmax(f1s)]:.4f}')

### Importance des features (Random Forest)

In [ ]:
rf_pipeline  = trained_pipes['Random Forest']
preprocessor = rf_pipeline.named_steps['pre']
rf_model     = rf_pipeline.named_steps['clf']

cat_feature_names = preprocessor.named_transformers_['cat'].get_feature_names_out(cat_cols).tolist()
all_feature_names = cat_feature_names + num_cols

feat_imp_df = pd.DataFrame({
    'Feature':    all_feature_names,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=True)

imp_colors = ['#e74c3c' if v > 0.1 else '#3498db' for v in feat_imp_df['Importance']]

plt.figure(figsize=(10, 6))
plt.barh(feat_imp_df['Feature'], feat_imp_df['Importance'], color=imp_colors)
plt.axvline(x=0.1, color='red', linestyle='--', alpha=0.5, label='Seuil 10%')
plt.title('Importance des Features – Random Forest', fontsize=12, fontweight='bold')
plt.xlabel('Importance')
plt.legend()
plt.tight_layout()
plt.show()

print('Top 5 features :')
print(feat_imp_df.sort_values('Importance', ascending=False).head(5).to_string(index=False))

## Conclusion

### Comparaison des modèles

| Modèle | Forces | Faiblesses |
|---|---|---|
| **Logistic Regression** | Rapide, interprétable, calibrée | Frontière linéaire uniquement |
| **KNN (k=5)** | Simple, non-paramétrique | Lent sur 100k lignes, sensible à l'échelle |
| **Random Forest** | Meilleure performance, robuste | Moins interprétable, plus lourd |

### Recommandation médicale

Le **Recall** est la métrique prioritaire : manquer un diabétique (faux négatif) est plus grave qu'une fausse alarme.  
**Modèle recommandé :** Random Forest avec seuil abaissé à ~0.30–0.35 pour maximiser le rappel.

### Features les plus prédictives
1. `HbA1c_level` — marqueur clé du diabète chronique
2. `blood_glucose_level` — glycémie directe
3. `bmi` — indice de masse corporelle
4. `age` — le risque augmente avec l'âge